# Codex Rollout Log Analysis

This notebook loads the rollout JSONL log and provides quick diagnostics:
- event volume over time
- top event types
- assistant/user message distribution
- task/tool execution summaries

In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_colwidth', 160)
plt.style.use('ggplot')

In [ ]:
LOG_PATH = Path('rollout-2026-03-31T13-25-09-019d43a3-eaf9-72b0-9133-6dcc03290f40.jsonl')
if not LOG_PATH.exists():
    alt = Path('generation_analysis') / LOG_PATH.name
    if alt.exists():
        LOG_PATH = alt

rows = []
with LOG_PATH.open('r', encoding='utf-8') as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            rows.append({'line_number': line_number, 'decode_error': True, 'raw': line})
            continue

        payload = item.get('payload') if isinstance(item.get('payload'), dict) else {}
        content = item.get('content') if isinstance(item.get('content'), dict) else {}

        rows.append({
            'line_number': line_number,
            'timestamp': item.get('timestamp'),
            'type': item.get('type'),
            'payload_type': payload.get('type'),
            'payload_role': payload.get('role'),
            'payload_name': payload.get('name'),
            'payload_call_id': payload.get('call_id'),
            'payload_status': payload.get('status'),
            'content_type': content.get('type'),
            'content_level': content.get('level'),
            'decode_error': False,
            'raw_obj': item,
        })

df = pd.DataFrame(rows)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)

print(f'Loaded {len(df):,} log lines from: {LOG_PATH}')
df.head()

In [ ]:
summary = {
    'rows': len(df),
    'time_start': df['timestamp'].min(),
    'time_end': df['timestamp'].max(),
    'decode_errors': int(df['decode_error'].sum()) if 'decode_error' in df.columns else 0,
}
pd.Series(summary)

In [ ]:
print('Top high-level event types')
display(df['type'].value_counts().head(20).to_frame('count'))

print('Top payload event types')
display(df['payload_type'].fillna('<none>').value_counts().head(20).to_frame('count'))

print('Message roles')
display(df['payload_role'].fillna('<none>').value_counts().to_frame('count'))

In [ ]:
timeline = (
    df.dropna(subset=['timestamp'])
      .set_index('timestamp')
      .resample('10S')
      .size()
      .rename('events')
)

ax = timeline.plot(figsize=(12, 4), title='Events over time (10s bins)')
ax.set_ylabel('event count')
ax.set_xlabel('timestamp (UTC)')
plt.tight_layout()
plt.show()

In [ ]:
tool_calls = df[df['payload_type'].isin(['function_call', 'custom_tool_call'])].copy()
tool_results = df[df['payload_type'].isin(['function_call_output', 'custom_tool_call_output'])].copy()

print(f'Tool call events: {len(tool_calls):,}')
print(f'Tool output events: {len(tool_results):,}')

if len(tool_calls):
    display(tool_calls[['timestamp', 'payload_type', 'payload_name', 'payload_call_id']].head(20))

if len(tool_results):
    display(tool_results[['timestamp', 'payload_type', 'payload_call_id', 'payload_status']].head(20))